Importing

In [2]:
import pandas as pd

df = pd.read_csv("news_summary_more.csv",engine="python", on_bad_lines='skip')
df.head()

,headlines,text
0,upGrad learner switches to career in ML & Al w...,"Saurav Kant, an alumnus of upGrad and IIIT-B's..."
1,Delhi techie wins free food from Swiggy for on...,Kunal Shah's credit card bill payment platform...
2,New Zealand end Rohit Sharma-led India's 12-ma...,New Zealand defeated India by 8 wickets in the...
3,Aegon life iTerm insurance plan helps customer...,"With Aegon Life iTerm Insurance plan, customer..."
4,"Have known Hirani for yrs, what if MeToo claim...",Speaking about the sexual harassment allegatio...


In [3]:
df = df.rename(columns={
    "text": "article",
    "headlines": "summary"
})

In [4]:
df.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98401 entries, 0 to 98400
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   summary  98401 non-null  object
 1   article  98401 non-null  object
dtypes: object(2)
memory usage: 1.5+ MB


,summary,article
0,upGrad learner switches to career in ML & Al w...,"Saurav Kant, an alumnus of upGrad and IIIT-B's..."
1,Delhi techie wins free food from Swiggy for on...,Kunal Shah's credit card bill payment platform...
2,New Zealand end Rohit Sharma-led India's 12-ma...,New Zealand defeated India by 8 wickets in the...
3,Aegon life iTerm insurance plan helps customer...,"With Aegon Life iTerm Insurance plan, customer..."
4,"Have known Hirani for yrs, what if MeToo claim...",Speaking about the sexual harassment allegatio...
...,...,...
98396,CRPF jawan axed to death by Maoists in Chhatti...,A CRPF jawan was on Tuesday axed to death with...
98397,First song from Sonakshi Sinha's 'Noor' titled...,"'Uff Yeh', the first song from the Sonakshi Si..."
98398,'The Matrix' film to get a reboot: Reports,"According to reports, a new version of the 199..."
98399,Snoop Dogg aims gun at clown dressed as Trump ...,A new music video shows rapper Snoop Dogg aimi...


Remove null & duplicates

In [5]:
df = df.dropna()
df = df.drop_duplicates()


Limit article & summary length

(Helps memory + faster training)

In [14]:
df['article_len'] = df['article'].apply(lambda x: len(x.split()))
df['summary_len'] = df['summary'].apply(lambda x: len(x.split()))

df = df[(df['article_len'] <= 300) & (df['summary_len'] <= 40)]
df

,summary,article,article_len,summary_len
37796,<sos> exdentists new mobile app gets mn after ...,former south korean dentist seunggun lees mobi...,57,11
79336,<sos> lg tests robots to guide passengers in a...,lg is testing two robot prototypes to provide ...,59,12
9158,<sos> opposition is trying to kill me delhi cm...,reacting to the chilli powder attack on him on...,60,14
12600,<sos> policeman touches up ministers feet for ...,a policeman in uttar pradeshs kanpur touched t...,55,11
94005,<sos> id love to make mahabharat film but dont...,actor shah rukh khan has said that he would lo...,60,13
...,...,...,...,...
94787,<sos> the highest library book fine paid is <eos>,the worlds largest fine for an overdue library...,56,9
58728,<sos> delhi lg expected to show constitutional...,the supreme court has said that the lieutenant...,58,10
77358,<sos> startup that delivers fuel to parking lo...,usbased startup booster that delivers fuel to ...,58,11
51247,<sos> plane makes sideways landing after being...,a video footage shows a propeller plane succes...,55,12


Random Sampling

In [7]:
df = df.sample(n=10000, random_state=42)


Text Cleaning

In [8]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['article'] = df['article'].apply(clean_text)
df['summary'] = df['summary'].apply(clean_text)


Add Start & End Tokens (for Seq2Seq)

In [9]:
df['summary'] = "<sos> " + df['summary'] + " <eos>"


Tokenization

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 30000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<unk>")
tokenizer.fit_on_texts(df['article'].tolist() + df['summary'].tolist())


c:\Users\lenovo\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Convert Text → Sequences

In [11]:
article_seq = tokenizer.texts_to_sequences(df['article'])
summary_seq = tokenizer.texts_to_sequences(df['summary'])


Padding(Fixed Length)

In [12]:
ARTICLE_MAX_LEN = 300
SUMMARY_MAX_LEN = 40

# Re-pad sequences
encoder_input = pad_sequences(
    article_seq,
    maxlen=ARTICLE_MAX_LEN,
    padding='post'
)

decoder_input = pad_sequences(
    summary_seq,
    maxlen=SUMMARY_MAX_LEN,
    padding='post'
)

# Shift decoder
decoder_target = decoder_input[:, 1:]
decoder_input = decoder_input[:, :-1]

Tokenized & padded view

In [ ]:
encoder_input[i][:20]
decoder_input[i][:20]


Train–Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

enc_train, enc_val, dec_train, dec_val, tgt_train, tgt_val = train_test_split(
    encoder_input,
    decoder_input,
    decoder_target,
    test_size=0.1,
    random_state=42
)


Model Definition (Seq2Seq + Attention)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, LSTM, Embedding, Dense,
    Attention, Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


In [ ]:
VOCAB_SIZE = 30000
EMB_DIM = 128
LSTM_UNITS = 100
ARTICLE_MAX_LEN = 300
SUMMARY_MAX_LEN = 40


ENCODER

In [ ]:
encoder_inputs = Input(shape=(ARTICLE_MAX_LEN,))
enc_emb = Embedding(
    VOCAB_SIZE,
    EMB_DIM,
    mask_zero=False
)(encoder_inputs)

encoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True,
    dropout=0.35,
    recurrent_dropout=0.0
)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)


DECODER

In [ ]:
decoder_inputs = Input(shape=(SUMMARY_MAX_LEN - 1,))
dec_emb = Embedding(
    VOCAB_SIZE,
    EMB_DIM,
    mask_zero=False
)(decoder_inputs)

decoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True,
    dropout=0.3,
    recurrent_dropout=0.3
)

decoder_outputs, _, _ = decoder_lstm(
    dec_emb,
    initial_state=[state_h, state_c]
)


LUONG ATTENTION

In [ ]:


from tensorflow.keras.layers import Attention, Concatenate

luong_attention = Attention(name="luong_attention")
attn_output = luong_attention([decoder_outputs, encoder_outputs])

concat = Concatenate(axis=-1)([decoder_outputs, attn_output])



BAHDANAU ATTENTION

In [ ]:
from tensorflow.keras.layers import Layer, Dense
import tensorflow as tf

class BahdanauAttention(Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, encoder_outputs, decoder_outputs):
        # encoder_outputs: (batch, enc_len, units)
        # decoder_outputs: (batch, dec_len, units)

        enc_exp = tf.expand_dims(encoder_outputs, 1)
        dec_exp = tf.expand_dims(decoder_outputs, 2)

        score = self.V(
            tf.nn.tanh(
                self.W1(enc_exp) + self.W2(dec_exp)
            )
        )

        attention_weights = tf.nn.softmax(score, axis=2)
        context_vector = attention_weights * enc_exp
        context_vector = tf.reduce_sum(context_vector, axis=2)

        return context_vector

# Instantiate and apply Bahdanau attention
bahdanau_attention = BahdanauAttention(LSTM_UNITS)
context_vector = bahdanau_attention(encoder_outputs, decoder_outputs)

# Concatenate decoder outputs with the context vector
concat = Concatenate(axis=-1)([decoder_outputs, context_vector])

OUTPUT LAYER

In [ ]:
decoder_dense = Dense(VOCAB_SIZE, activation="softmax")
decoder_outputs = decoder_dense(concat)

FINAL MODEL

In [ ]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


EARLY STOPPING

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)


TRAINING

In [ ]:

history = model.fit(
    [enc_train, dec_train],
    tgt_train,
    validation_data=([enc_val, dec_val], tgt_val),
    batch_size=32,
    epochs=10,
    callbacks=[early_stop]
)


In [ ]:
!pip install rouge-score

Rouge Evaluaton

In [ ]:
import numpy as np
from rouge_score import rouge_scorer

Helper: tokens → sentence

In [ ]:
def tokens_to_text(tokens):
    words = []
    for tok in tokens:
        if tok == 0:
            continue
        word = tokenizer.index_word.get(tok, "")
        if word in ["<sos>", "<eos>"]:
            continue
        words.append(word)
    return " ".join(words)


Predict summaries

In [ ]:
# Predict on validation set
pred_probs = model.predict([enc_val, dec_val], batch_size=32)

# Convert probabilities to token IDs
pred_tokens = np.argmax(pred_probs, axis=-1)


Initialize ROUGE and Compute ROUGE scores

In [ ]:
r1, r2, rL = [], [], []

N = 50  # evaluate on first 50 validation samples


scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

for i in range(N):
    # Ground truth
    ref_text = tokens_to_text(tgt_val[i])

    # Predicted summary
    pred_text = tokens_to_text(pred_tokens[i])

    if len(pred_text.split()) < 3:
        continue

    scores = scorer.score(ref_text, pred_text)

    r1.append(scores['rouge1'].fmeasure)
    r2.append(scores['rouge2'].fmeasure)
    rL.append(scores['rougeL'].fmeasure)


In [ ]:
print("Avg ROUGE-1:", np.mean(r1))
print("Avg ROUGE-2:", np.mean(r2))
print("Avg ROUGE-L:", np.mean(rL))


Training vs Validation loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()
